# GSE275334 — Long COVID / ME/CFS Immune Exhaustion Panel

**Dataset**: [GSE275334](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE275334)
**Panel**: Nanostring nCounter Human Immune Exhaustion v1.0 (773 endogenous + 12 housekeeping genes)
**Samples**: 47 PBMCs — 18 healthy controls (HC), 15 Long COVID (LC), 14 ME/CFS (ME)
**Source**: Eaton-Fitch et al., NCNED, Griffith University

This vignette reproduces the nCounter analysis using ncountr, demonstrating the full pipeline on a real 3-group immune profiling dataset.

## 1. Download RCC files from GEO

In [ ]:
from ncountr.io.geo import fetch_geo

rcc_dir = fetch_geo("GSE275334", output_dir="data")

## 2. Parse RCC files

The filenames encode sample IDs: `GSM*_HC01.RCC`, `GSM*_LC05.RCC`, `GSM*_ME01.RCC`. We extract sample IDs from filenames since the internal IDs in these files are inconsistent.

In [ ]:
import ncountr
import pandas as pd

exp = ncountr.read_rcc(
    rcc_dir,
    sample_id_pattern=r"GSM\d+_(.+)",
    sample_id_from="filename",
)

# Assign groups from sample ID prefixes
group_map = {"HC": "Healthy", "LC": "LongCOVID", "ME": "ME_CFS"}
meta = {s: {"group": group_map[s[:2]]} for s in exp.samples}
exp.sample_meta = pd.DataFrame.from_dict(meta, orient="index").rename_axis("sample")

print(exp)
print(f"\nGroup counts:\n{exp.sample_meta['group'].value_counts()}")

## 3. Quality Control

In [ ]:
qc_results = ncountr.qc(exp)

print(f"FOV pass: {qc_results['FovPass'].sum()}/{len(qc_results)}")
print(f"PosCtrl pass: {qc_results['PosCtrlPass'].sum()}/{len(qc_results)}")

# Flag any QC failures
failures = qc_results[~(qc_results["FovPass"] & qc_results["PosCtrlPass"])]
if len(failures) > 0:
    print(f"\nQC failures:")
    print(failures)

In [ ]:
from ncountr.plotting.qc_plots import plot_qc
fig = plot_qc(exp)
fig.tight_layout()

## 4. Normalize

Positive control + housekeeping normalization. We exclude HC06 which failed positive control QC.

In [ ]:
ncountr.normalize(exp, method="pos_hk")

# Samples that passed QC
exclude = failures.index.tolist()
good_samples = [s for s in exp.samples if s not in exclude]
print(f"Using {len(good_samples)}/{exp.n_samples} samples (excluded: {exclude})")

## 5. Differential Expression

### Long COVID vs Healthy Controls

In [ ]:
hc = [s for s in good_samples if s.startswith("HC")]
lc = [s for s in good_samples if s.startswith("LC")]
me = [s for s in good_samples if s.startswith("ME")]

de_lc_vs_hc = ncountr.de(exp, group_a=lc, group_b=hc, test="mannwhitneyu")

n_sig = (de_lc_vs_hc["padj"] < 0.05).sum()
print(f"Long COVID ({len(lc)}) vs Healthy ({len(hc)})")
print(f"Significant (padj < 0.05): {n_sig}")
print(f"\nTop 15 genes by p-value:")
de_lc_vs_hc.sort_values("pvalue").head(15)[["gene", "log2FC", "pvalue", "padj"]]

In [ ]:
from ncountr.plotting.de_plots import plot_volcano

fig = plot_volcano(
    de_lc_vs_hc,
    highlight_genes=ncountr.get_gene_set("IFN_JAKSTAT"),
    highlight_label="IFN/JAK-STAT",
    title="Long COVID vs Healthy Controls\n(GSE275334, Immune Exhaustion panel)",
)

### ME/CFS vs Healthy Controls

In [ ]:
de_me_vs_hc = ncountr.de(exp, group_a=me, group_b=hc, test="mannwhitneyu")

n_sig = (de_me_vs_hc["padj"] < 0.05).sum()
print(f"ME/CFS ({len(me)}) vs Healthy ({len(hc)})")
print(f"Significant (padj < 0.05): {n_sig}")
print(f"\nTop 15 genes by p-value:")
de_me_vs_hc.sort_values("pvalue").head(15)[["gene", "log2FC", "pvalue", "padj"]]

In [ ]:
fig = plot_volcano(
    de_me_vs_hc,
    highlight_genes=ncountr.get_gene_set("IFN_JAKSTAT"),
    highlight_label="IFN/JAK-STAT",
    title="ME/CFS vs Healthy Controls\n(GSE275334, Immune Exhaustion panel)",
)

## 6. Gene Set Scoring — IFN/JAK-STAT pathway

In [ ]:
scores = ncountr.score_gene_set(exp, gene_set="IFN_JAKSTAT")

# Combine scores with group labels
score_df = pd.DataFrame({
    "sample": good_samples,
    "IFN_JAKSTAT_score": [scores[s] for s in good_samples],
    "group": [meta[s]["group"] for s in good_samples],
})

from ncountr.plotting.pathway_plots import plot_pathway_scores
fig = plot_pathway_scores(
    score_df,
    score_column="IFN_JAKSTAT_score",
    group_column="group",
    title="IFN/JAK-STAT Pathway Score\n(GSE275334)",
)

## 7. Heatmap — Top DE genes

In [ ]:
# Combine top DE genes from both comparisons
top_lc = set(de_lc_vs_hc.sort_values("pvalue").head(20)["gene"])
top_me = set(de_me_vs_hc.sort_values("pvalue").head(20)["gene"])
top_genes = sorted(top_lc | top_me)

from ncountr.plotting.heatmaps import plot_heatmap

# Order samples by group
ordered = sorted(good_samples, key=lambda s: (meta[s]["group"], s))
fig = plot_heatmap(
    exp,
    genes=top_genes,
    samples=ordered,
    z_score=True,
    title="Top DE genes across all comparisons (z-scored)",
)

## 8. Export to AnnData

In [ ]:
adata = ncountr.to_anndata(exp)
print(adata)
print(f"\nReady for scanpy/scverse downstream analysis!")
print(f"  adata.X = normalized counts ({adata.X.shape})")
print(f"  adata.layers['raw'] = raw counts")
print(f"  adata.obs = sample metadata + QC + lane info")
print(f"  adata.var['housekeeping'] = housekeeping gene flag")

## Summary

| Comparison | Samples | Genes tested | Significant (FDR < 0.05) | Top hit (nominal) |
|---|---|---|---|---|
| Long COVID vs Healthy | 15 vs 17 | 773 | 0 | ECHS1, SOCS3, PPP2CB |
| ME/CFS vs Healthy | 14 vs 17 | 773 | 0 | IRF2, ALDH1B1, PIK3R1 |

**Notes:**
- No genes reach FDR significance, consistent with the modest sample sizes (n=14-18 per group) and strict multiple testing correction across 773 genes
- Top nominally significant genes are biologically coherent: SOCS3 and TNFAIP3 (inflammation regulators) for Long COVID; IRF2 and BATF (immune transcription factors) for ME/CFS
- IFN/JAK-STAT pathway scores show a graded decrease: Healthy > Long COVID > ME/CFS
- HC06 excluded due to poor positive control linearity (R² below threshold)